In [1]:
import logging
import re
from abc import ABC, abstractmethod
from datetime import datetime

logging.basicConfig(
    filename="sistema_software_fj.log",
    filemode="w",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True
)

class ErrorSistemaFJ(Exception):
    """Error general del sistema."""
    pass

class ErrorValidacionCliente(ErrorSistemaFJ):
    """Error cuando algún dato del cliente está mal."""
    pass

class ErrorParametrosServicio(ErrorSistemaFJ):
    """Error por datos faltantes o inválidos al crear un servicio."""
    pass

class ErrorServicioNoDisponible(ErrorSistemaFJ):
    """Error al intentar reservar un servicio que no está habilitado."""
    pass

class ErrorReservaInvalida(ErrorSistemaFJ):
    """Error en operaciones no permitidas sobre una reserva."""
    pass

class ErrorCalculoCosto(ErrorSistemaFJ):
    """Error cuando hay inconsistencias al calcular el valor total."""
    pass

logging.info("Inicio del sistema y configuración de logs.")

In [2]:
class EntidadSistema(ABC):
    """Clase base abstracta para las entidades del sistema."""

    def __init__(self, entidad_id: str):
        if not entidad_id or not isinstance(entidad_id, str):
            raise ValueError("El ID debe ser un texto válido y no estar vacío.")
        self._id = entidad_id.strip()
        self._fecha_creacion = datetime.now()

    @property
    def id(self) -> str:
        return self._id

    @abstractmethod
    def obtener_resumen(self) -> str:
        """Retorna un resumen con los datos principales."""
        pass


class Cliente(EntidadSistema):
    """Gestiona los datos y validaciones de un cliente."""

    def __init__(self, cliente_id: str, nombre: str, documento: str, correo: str, telefono: str):
        super().__init__(cliente_id)
        self.nombre = nombre
        self.documento = documento
        self.correo = correo
        self.telefono = telefono
        logging.info(f"Cliente registrado: {self._id} - {self.__nombre}")

    @property
    def nombre(self) -> str:
        return self.__nombre

    @nombre.setter
    def nombre(self, valor: str):
        if not valor or not isinstance(valor, str) or len(valor.strip()) < 3:
            raise ErrorValidacionCliente("El nombre debe tener al menos 3 caracteres.")
        self.__nombre = valor.strip()

    @property
    def documento(self) -> str:
        return self.__documento

    @documento.setter
    def documento(self, valor: str):
        if not valor or not str(valor).isdigit() or len(str(valor)) < 6:
            raise ErrorValidacionCliente("El documento debe ser numérico y tener mínimo 6 dígitos.")
        self.__documento = str(valor)

    @property
    def correo(self) -> str:
        return self.__correo

    @correo.setter
    def correo(self, valor: str):
        patron_correo = r"^[\w\.-]+@[\w\.-]+\.\w+$"
        if not valor or not re.match(patron_correo, valor.strip()):
            raise ErrorValidacionCliente(f"El correo '{valor}' no es válido.")
        self.__correo = valor.strip()

    @property
    def telefono(self) -> str:
        return self.__telefono

    @telefono.setter
    def telefono(self, valor: str):
        if not valor or not str(valor).isdigit() or len(str(valor)) < 7:
            raise ErrorValidacionCliente("El teléfono debe contener solo números y mínimo 7 dígitos.")
        self.__telefono = str(valor)

    def obtener_resumen(self) -> str:
        return f"Cliente [{self._id}] {self.__nombre} | Doc: {self.__documento} | Correo: {self.__correo}"

In [3]:
class Servicio(EntidadSistema):
    """Clase base abstracta para los servicios de Software FJ."""

    def __init__(self, servicio_id: str, nombre: str, tarifa_base: float, disponible: bool = True):
        super().__init__(servicio_id)
        self._nombre = nombre
        self._tarifa_base = tarifa_base
        self._disponible = disponible
        self.validar_parametros()

    @property
    def nombre(self) -> str:
        return self._nombre

    @property
    def disponible(self) -> bool:
        return self._disponible

    @disponible.setter
    def disponible(self, estado: bool):
        self._disponible = bool(estado)

    @abstractmethod
    def validar_parametros(self) -> None:
        """Verifica que los datos del servicio sean válidos."""
        pass

    @abstractmethod
    def calcular_costo(self, duracion_horas: float, impuesto: float = 0.0, descuento: float = 0.0) -> float:
        """Calcula el valor total con impuestos y descuentos opcionales."""
        pass

    @abstractmethod
    def describir(self) -> str:
        """Retorna la descripción del servicio."""
        pass

    def obtener_resumen(self) -> str:
        estado = "Disponible" if self._disponible else "No disponible"
        return f"Servicio [{self._id}] {self._nombre} ({estado})"


class ReservaSala(Servicio):
    """Servicio de reserva de salas."""

    def __init__(self, servicio_id: str, nombre: str, tarifa_base: float, capacidad: int, tiene_proyector: bool = True, disponible: bool = True):
        self._capacidad = capacidad
        self._tiene_proyector = tiene_proyector
        super().__init__(servicio_id, nombre, tarifa_base, disponible)
        logging.info(f"Servicio ReservaSala creado: {self._id} - {self._nombre}")

    def validar_parametros(self) -> None:
        if self._tarifa_base <= 0:
            raise ErrorParametrosServicio("La tarifa base de la sala debe ser mayor a 0.")
        if self._capacidad <= 0 or self._capacidad > 200:
            raise ErrorParametrosServicio("La capacidad de la sala debe estar entre 1 y 200 personas.")

    def calcular_costo(self, duracion_horas: float, impuesto: float = 0.19, descuento: float = 0.0) -> float:
        if duracion_horas <= 0:
            raise ErrorCalculoCosto("La duración de la reserva de sala debe ser mayor a 0 horas.")
        if not (0 <= descuento <= 0.5):
            raise ErrorCalculoCosto("El descuento debe estar entre 0% (0.0) y 50% (0.5).")
        subtotal = self._tarifa_base * duracion_horas
        if self._tiene_proyector:
            subtotal += 20000
        subtotal_con_descuento = subtotal * (1 - descuento)
        return round(subtotal_con_descuento * (1 + impuesto), 2)

    def describir(self) -> str:
        proyector = "con proyector" if self._tiene_proyector else "sin proyector"
        return f"Sala '{self._nombre}' (Capacidad: {self._capacidad} personas, {proyector}) - Tarifa: ${self._tarifa_base}/h"


class AlquilerEquipo(Servicio):
    """Servicio de alquiler de equipos tecnológicos."""

    def __init__(self, servicio_id: str, nombre: str, tarifa_base: float, tipo_equipo: str, cantidad: int = 1, disponible: bool = True):
        self._tipo_equipo = tipo_equipo
        self._cantidad = cantidad
        super().__init__(servicio_id, nombre, tarifa_base, disponible)
        logging.info(f"Servicio AlquilerEquipo creado: {self._id} - {self._nombre}")

    def validar_parametros(self) -> None:
        if self._tarifa_base <= 0:
            raise ErrorParametrosServicio("La tarifa base del equipo debe ser mayor a 0.")
        if self._cantidad <= 0:
            raise ErrorParametrosServicio("La cantidad de equipos debe ser al menos 1.")
        if not self._tipo_equipo or not isinstance(self._tipo_equipo, str):
            raise ErrorParametrosServicio("Debe especificar un tipo de equipo válido.")

    def calcular_costo(self, duracion_horas: float, impuesto: float = 0.19, descuento: float = 0.0, seguro: float = 0.05) -> float:
        if duracion_horas <= 0:
            raise ErrorCalculoCosto("Las horas de alquiler deben ser mayores a 0.")
        subtotal = (self._tarifa_base * self._cantidad) * duracion_horas
        subtotal_desc = subtotal * (1 - descuento)
        total = subtotal_desc * (1 + impuesto + seguro)
        return round(total, 2)

    def describir(self) -> str:
        return f"Alquiler de {self._cantidad}x {self._tipo_equipo} ('{self._nombre}') - Tarifa: ${self._tarifa_base}/h"


class AsesoriaEspecializada(Servicio):
    """Servicio de asesorías técnicas."""

    def __init__(self, servicio_id: str, nombre: str, tarifa_base: float, especialidad: str, modalidad: str = "Virtual", disponible: bool = True):
        self._especialidad = especialidad
        self._modalidad = modalidad
        super().__init__(servicio_id, nombre, tarifa_base, disponible)
        logging.info(f"Servicio AsesoriaEspecializada creado: {self._id} - {self._nombre}")

    def validar_parametros(self) -> None:
        if self._tarifa_base <= 0:
            raise ErrorParametrosServicio("La tarifa de la asesoría debe ser mayor a 0.")
        if self._modalidad not in ("Virtual", "Presencial"):
            raise ErrorParametrosServicio("La modalidad debe ser 'Virtual' o 'Presencial'.")

    def calcular_costo(self, duracion_horas: float, impuesto: float = 0.19, descuento: float = 0.0, urgente: bool = False) -> float:
        if duracion_horas <= 0:
            raise ErrorCalculoCosto("La duración de la asesoría debe ser mayor a 0.")
        subtotal = self._tarifa_base * duracion_horas
        if urgente:
            subtotal *= 1.25
        subtotal_desc = subtotal * (1 - descuento)
        return round(subtotal_desc * (1 + impuesto), 2)

    def describir(self) -> str:
        return f"Asesoría en {self._especialidad} ({self._modalidad}) - Tarifa: ${self._tarifa_base}/h"

In [4]:
class Reserva(EntidadSistema):
    """Gestiona la reserva de un servicio por parte de un cliente."""

    def __init__(self, reserva_id: str, cliente: Cliente, servicio: Servicio, duracion_horas: float):
        super().__init__(reserva_id)
        if not isinstance(cliente, Cliente):
            raise ErrorReservaInvalida("La reserva requiere un objeto Cliente válido.")
        if not isinstance(servicio, Servicio):
            raise ErrorReservaInvalida("La reserva requiere un objeto Servicio válido.")
        if duracion_horas <= 0:
            raise ErrorReservaInvalida("La duración de la reserva debe ser mayor a 0 horas.")

        self._cliente = cliente
        self._servicio = servicio
        self._duracion_horas = duracion_horas
        self._estado = "PENDIENTE"
        self._costo_total = 0.0

    @property
    def estado(self) -> str:
        return self._estado

    def confirmar(self, impuesto: float = 0.19, descuento: float = 0.0) -> float:
        """Confirma la reserva y calcula el valor total."""
        try:
            if not self._servicio.disponible:
                raise ErrorServicioNoDisponible(f"El servicio '{self._servicio.nombre}' no está disponible.")
            if self._estado != "PENDIENTE":
                raise ErrorReservaInvalida(f"No se puede confirmar una reserva en estado '{self._estado}'.")
            costo = self._servicio.calcular_costo(self._duracion_horas, impuesto=impuesto, descuento=descuento)
        except ErrorCalculoCosto as e:
            logging.error(f"Error de cálculo en reserva {self._id}: {e}")
            raise ErrorReservaInvalida(f"Fallo al confirmar la reserva {self._id} por error de cálculo.") from e
        except ErrorSistemaFJ as e:
            logging.error(f"Error al confirmar reserva {self._id}: {e}")
            raise
        else:
            self._costo_total = costo
            self._estado = "CONFIRMADA"
            logging.info(f"Reserva {self._id} CONFIRMADA | Cliente: {self._cliente.nombre} | Total: ${self._costo_total}")
            return self._costo_total

    def cancelar(self, motivo: str = "Cancelación solicitada por el cliente") -> None:
        """Cancela la reserva y registra el motivo en el log."""
        try:
            if self._estado == "CANCELADA":
                raise ErrorReservaInvalida(f"La reserva {self._id} ya se encuentra cancelada.")
            self._estado = "CANCELADA"
            logging.info(f"Reserva {self._id} CANCELADA. Motivo: {motivo}")
        except ErrorReservaInvalida as e:
            logging.warning(f"Intento inválido de cancelación en {self._id}: {e}")
            raise
        finally:
            logging.info(f"Cierre de proceso de cancelación para la reserva {self._id} (Estado actual: {self._estado}).")

    def procesar(self) -> str:
        """Procesa una reserva que ya fue confirmada."""
        try:
            if self._estado != "CONFIRMADA":
                raise ErrorReservaInvalida(f"Solo se pueden procesar reservas CONFIRMADAS (estado actual: {self._estado}).")
            self._estado = "PROCESADA"
            logging.info(f"Reserva {self._id} PROCESADA exitosamente.")
            return f"Reserva {self._id} procesada correctamente por ${self._costo_total}."
        except ErrorReservaInvalida as e:
            logging.error(f"Error al procesar reserva {self._id}: {e}")
            raise

    def obtener_resumen(self) -> str:
        return (
            f"Reserva [{self._id}] | Estado: {self._estado} | "
            f"Cliente: {self._cliente.nombre} | Servicio: {self._servicio.nombre} | "
            f"Horas: {self._duracion_horas} | Total: ${self._costo_total}"
        )

In [5]:
class SistemaSoftwareFJ:
    """Gestiona los clientes, servicios y reservas en memoria."""

    def __init__(self):
        self.clientes = []
        self.servicios = []
        self.reservas = []

    def registrar_cliente(self, cliente_id, nombre, documento, correo, telefono):
        try:
            cliente = Cliente(cliente_id, nombre, documento, correo, telefono)
            self.clientes.append(cliente)
            print(f"[OK] {cliente.obtener_resumen()}")
            return cliente
        except (ErrorValidacionCliente, ValueError) as e:
            logging.error(f"Fallo en registro de cliente ({cliente_id}): {e}")
            print(f"[ERROR CONTROLADO] Registro de cliente fallido: {e}")
            return None

    def registrar_servicio(self, servicio_clase, *args, **kwargs):
        try:
            servicio = servicio_clase(*args, **kwargs)
            self.servicios.append(servicio)
            print(f"[OK] {servicio.describir()}")
            return servicio
        except (ErrorParametrosServicio, ValueError, TypeError) as e:
            logging.error(f"Fallo en creación de servicio: {e}")
            print(f"[ERROR CONTROLADO] Creación de servicio fallida: {e}")
            return None

    def crear_y_confirmar_reserva(self, reserva_id, cliente, servicio, duracion, impuesto=0.19, descuento=0.0):
        try:
            reserva = Reserva(reserva_id, cliente, servicio, duracion)
            reserva.confirmar(impuesto=impuesto, descuento=descuento)
            self.reservas.append(reserva)
            print(f"[OK] {reserva.obtener_resumen()}")
            return reserva
        except ErrorSistemaFJ as e:
            causa = f" (Causa original: {e.__cause__})" if e.__cause__ else ""
            logging.error(f"Fallo en reserva {reserva_id}: {e}{causa}")
            print(f"[ERROR CONTROLADO] Reserva {reserva_id} fallida: {e}{causa}")
            return None

In [6]:
sistema = SistemaSoftwareFJ()

print("--- Inicio de pruebas: Software FJ ---\n")

c1 = sistema.registrar_cliente("C01", "Laura Martínez", "1023456789", "laura.m@correo.com", "3001234567")
c2 = sistema.registrar_cliente("C02", "Carlos Gómez", "1098765432", "carlos.g@empresa.co", "3119876543")
c_invalido1 = sistema.registrar_cliente("C03", "Ana Ruiz", "10123456", "correo_sin_arroba", "3201112233")
c_invalido2 = sistema.registrar_cliente("C04", "Pedro Díaz", "ABC123", "pedro@correo.com", "3150001122")

s1 = sistema.registrar_servicio(ReservaSala, "S01", "Sala Innovación", 80000, capacidad=20, tiene_proyector=True)
s2 = sistema.registrar_servicio(AlquilerEquipo, "S02", "Portátil Workstation", 35000, tipo_equipo="Laptop i9", cantidad=2)
s3 = sistema.registrar_servicio(AsesoriaEspecializada, "S03", "Arquitectura Cloud", 120000, especialidad="Cloud AWS", modalidad="Virtual", disponible=False)
s_invalido = sistema.registrar_servicio(ReservaSala, "S04", "Sala Fallida", -50000, capacidad=10)

r1 = sistema.crear_y_confirmar_reserva("R01", c1, s1, duracion=3, impuesto=0.19, descuento=0.10)
r2 = sistema.crear_y_confirmar_reserva("R02", c2, s2, duracion=5)
if r2:
    print(f"[OK] {r2.procesar()}")

r_fallida1 = sistema.crear_y_confirmar_reserva("R03", c1, s3, duracion=2)
r_fallida2 = sistema.crear_y_confirmar_reserva("R04", c2, s1, duracion=2, descuento=0.85)

if r1:
    r1.cancelar("Cambio de agenda del cliente")
    print(f"[OK] Estado final de R01: {r1.estado}")

print("\n--- Fin de la simulación ---")
print(f"Resumen -> Clientes: {len(sistema.clientes)} | Servicios: {len(sistema.servicios)} | Reservas: {len(sistema.reservas)}")

--- Inicio de pruebas: Software FJ ---

[OK] Cliente [C01] Laura Martínez | Doc: 1023456789 | Correo: laura.m@correo.com
[OK] Cliente [C02] Carlos Gómez | Doc: 1098765432 | Correo: carlos.g@empresa.co
[ERROR CONTROLADO] Registro de cliente fallido: El correo 'correo_sin_arroba' no es válido.
[ERROR CONTROLADO] Registro de cliente fallido: El documento debe ser numérico y tener mínimo 6 dígitos.
[OK] Sala 'Sala Innovación' (Capacidad: 20 personas, con proyector) - Tarifa: $80000/h
[OK] Alquiler de 2x Laptop i9 ('Portátil Workstation') - Tarifa: $35000/h
[OK] Asesoría en Cloud AWS (Virtual) - Tarifa: $120000/h
[ERROR CONTROLADO] Creación de servicio fallida: La tarifa base de la sala debe ser mayor a 0.
[OK] Reserva [R01] | Estado: CONFIRMADA | Cliente: Laura Martínez | Servicio: Sala Innovación | Horas: 3 | Total: $278460.0
[OK] Reserva [R02] | Estado: CONFIRMADA | Cliente: Carlos Gómez | Servicio: Portátil Workstation | Horas: 5 | Total: $434000.0
[OK] Reserva R02 procesada correctamen

In [8]:
print("--- Registro de eventos (sistema_software_fj.log) ---\n")
with open("sistema_software_fj.log", "r", encoding="utf-8") as archivo:
    print(archivo.read())

--- Registro de eventos (sistema_software_fj.log) ---

2026-11-21 21:45:33 | INFO | Inicio del sistema y configuración de logs.
2026-11-21 21:45:33 | INFO | Cliente registrado: C01 - Laura Martínez
2026-11-21 21:45:33 | INFO | Cliente registrado: C02 - Carlos Gómez
2026-11-21 21:45:33 | ERROR | Fallo en registro de cliente (C03): El correo 'correo_sin_arroba' no es válido.
2026-11-21 21:45:33 | ERROR | Fallo en registro de cliente (C04): El documento debe ser numérico y tener mínimo 6 dígitos.
2026-11-21 21:45:33 | INFO | Servicio ReservaSala creado: S01 - Sala Innovación
2026-11-21 21:45:33 | INFO | Servicio AlquilerEquipo creado: S02 - Portátil Workstation
2026-11-21 21:45:33 | INFO | Servicio AsesoriaEspecializada creado: S03 - Arquitectura Cloud
2026-11-21 21:45:33 | ERROR | Fallo en creación de servicio: La tarifa base de la sala debe ser mayor a 0.
2026-11-21 21:45:33 | INFO | Reserva R01 CONFIRMADA | Cliente: Laura Martínez | Total: $278460.0
2026-11-21 21:45:33 | INFO | Reserva